# Calculation exact convergence generalized magnitude in real data

20260829
Kyoko Kusano

Comparing the genmag values between previous version and currenct exact computation version

In [4]:
import sys, pathlib
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tables


here = pathlib.Path.cwd()
for candidate in (here, here / "work", *here.parents):
    if (candidate / "magnitude_limit.py").exists():
        sys.path.insert(0, str(candidate))
        break

from genmag_exact import magnitude_limit as ml
print(ml.__file__)

/home/jovyan/work/src/magnitude_limit.py


In [5]:
from qstr_dataset import paths
h5_path = paths.interim("amy_processeddata_all.h5")
dissim_path = paths.interim("dissim_all.h5")

with pd.HDFStore(h5_path, mode="r") as store:
    genmag = store["/metrics/genmag"]
    positive_definite_t = store["/metrics/positivedefinite"]


def sub_id(key):          # '/sub_10' -> 10
    return int(re.search(r"\d+", key).group())

with pd.HDFStore(dissim_path, mode="r") as store:
    keys = sorted(store.keys(), key=sub_id)
    dissim = {sub_id(k): store[k] for k in keys}

In [6]:
genmag_last = genmag.iloc[:, -1]
mask = ~np.isclose(genmag_last, genmag_last.round()) & genmag_last.notna()
print("non-integer plateau in previous calculation")
print(genmag_last[mask])

soi = genmag_last[mask].index.tolist()

non-integer plateau in previous calculation
0       9.100000
14      3.222222
21      6.923077
39     11.666667
45     14.777778
56      7.035088
60      8.333333
63     15.333333
74     15.666667
85     11.400000
99      9.400000
106     4.733333
111    12.545455
Name: 100000000.0, dtype: float64


In [74]:
for i, sub in enumerate(soi):
    D = dissim[sub].to_numpy()
    result = ml.exact_limit(D)
    new_ = result.limit
    new = new_.evalf(5)
    rank = result.rank_at_infinity

    prev = genmag_last[mask].loc[sub].item()

    print(f"sub {sub}: {prev:.5f} -> {new}, rank: {rank}")

sub 0: 9.10000 -> 7.8750, rank: 22


KeyboardInterrupt: 

In [91]:
remove = {14, 45}
soi_select = [x for x in soi if x not in remove]
soi_select

[0, 21, 39, 56, 60, 63, 74, 85, 99, 106, 111]

In [92]:
for i, sub in enumerate(soi_select):
    D = dissim[sub].to_numpy()
    result = ml.exact_limit(D)
    new_ = result.limit
    new = new_.evalf(6)
    rank = result.rank_at_infinity

    prev = genmag_last[mask].loc[sub].item()

    print(f"sub {sub}: {prev:.5f} -> {new}, rank: {rank}")

sub 0: 9.10000 -> 7.87500, rank: 22
sub 21: 6.92308 -> 6.92308, rank: 23
sub 39: 11.66667 -> 11.6667, rank: 23
sub 56: 7.03509 -> 8.04082, rank: 22
sub 60: 8.33333 -> 8.33333, rank: 23
sub 63: 15.33333 -> 15.3333, rank: 21
sub 74: 15.66667 -> 15.6667, rank: 19
sub 85: 11.40000 -> 11.4000, rank: 21
sub 99: 9.40000 -> 9.44444, rank: 21
sub 106: 4.73333 -> 4.73333, rank: 23
sub 111: 12.54545 -> 11.5455, rank: 22


In [96]:
sub = 45
D = dissim[sub].to_numpy()
result = ml.exact_limit(D)
new_ = result.limit
new = new_.evalf(6)
rank = result.rank_at_infinity

prev = genmag_last[mask].loc[sub].item()

print(f"sub {sub}: {prev:.5f} -> {new}, rank: {rank}")

ValueError: The all-ones vector is not in im A(infinity)

In [ ]:
[0, 56, 99, 111]
[45] # error
[14] # timeout

In [99]:
# test with integer participants

sub = 110

D = dissim[110].to_numpy()
result = ml.exact_limit(D)
new_ = result.limit
new = new_.evalf(6)
rank = result.rank_at_infinity

prev = genmag_last.loc[sub].item()

print(f"sub {sub}: {prev:.5f} -> {new}, rank: {rank}")

sub 110: 15.00000 -> 14.0000, rank: 19


## 全被験者(120名)を 8 並列で計算

`exact_limit` は 1 プロセスで 1 コアしか使わないため、被験者ごとに別プロセスへ
振り分けて 8 コアを埋める。1 名あたり 5 分でタイムアウト、エラーが出た場合も
そこで打ち切って次へ進む。結果は CSV に保存する。


In [7]:
# =====================================================================
# 全被験者を 8 並列で計算する
#   - 1 名あたり TIMEOUT_SEC (5分) を超えたら打ち切り、"timeout" と表示して次へ
#   - 計算中にエラーが出たらエラー内容を表示して次へ
#   - 表示も CSV も sub 番号順。計算自体は並列のまま進み、表示だけを並べ替える
#   - prev と結果が異なる被験者には印を付け、CSV に changed 列 (bool) を入れる
#
# 前提: 上のセルで dissim / genmag_last / ml が読み込まれていること
# =====================================================================
import multiprocessing as mp
import signal
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures.process import BrokenProcessPool

import sympy as sp

TIMEOUT_SEC = 5 * 60   # 1 被験者あたりの上限（秒）
MAX_WORKERS = 8        # 同時に走らせるプロセス数。メモリが足りなければ 4 に下げる

# prev と新しい値が「異なる」と判定する許容誤差
CHANGE_RTOL = 1e-4
CHANGE_ATOL = 1e-6

CHANGED_MARK = "  <<< CHANGED"


class _Timeout(Exception):
    pass


def _on_alarm(signum, frame):
    raise _Timeout


def solve_one(sub):
    """子プロセスで 1 名分を計算する。戻り値は pickle できる dict のみ。

    sympy の式はそのままプロセス間で受け渡さず、文字列にしてから返す。
    """
    signal.signal(signal.SIGALRM, _on_alarm)
    signal.alarm(TIMEOUT_SEC)
    started = time.perf_counter()
    try:
        result = ml.exact_limit(dissim[sub].to_numpy())
        return {
            "sub": sub,
            "status": "ok",
            "limit": str(result.limit),
            "rank": result.rank_at_infinity,
            "sec": time.perf_counter() - started,
            "error": None,
        }
    except _Timeout:
        return {"sub": sub, "status": "timeout", "limit": None, "rank": None,
                "sec": time.perf_counter() - started, "error": None}
    except Exception as exc:
        return {"sub": sub, "status": "error", "limit": None, "rank": None,
                "sec": time.perf_counter() - started,
                "error": f"{type(exc).__name__}: {exc}"}
    finally:
        signal.alarm(0)


def _enrich(res):
    """prev / 数値化した新しい値 / changed を res に足す。"""
    prev = genmag_last.get(res["sub"])
    prev = float(prev) if prev is not None and pd.notna(prev) else None
    res["prev"] = prev

    new_value = None
    if res["status"] == "ok":
        try:
            new_value = float(sp.sympify(res["limit"]))
        except (TypeError, ValueError):
            new_value = None      # sympy.oo など数値化できないもの
    res["limit_float"] = new_value

    # 判定できないときは None のままにして、False と区別する
    if prev is None or new_value is None:
        res["changed"] = None
    else:
        res["changed"] = not np.isclose(prev, new_value,
                                        rtol=CHANGE_RTOL, atol=CHANGE_ATOL)
    return res


def _report(res):
    """1 名分を 1 行で表示する。直前の進捗表示を上書きするため \r で始める。"""
    sub = res["sub"]
    prev_s = f"{res['prev']:8.5f}" if res["prev"] is not None else "     n/a"

    if res["status"] == "ok":
        new = sp.sympify(res["limit"]).evalf(6)
        mark = CHANGED_MARK if res["changed"] else ""
        line = (f"sub {sub:3d}: {prev_s} -> {new}, rank: {res['rank']}"
                f"   [{res['sec']:.1f}s]{mark}")
    elif res["status"] == "timeout":
        line = (f"sub {sub:3d}: {prev_s} -> timeout "
                f"({TIMEOUT_SEC // 60} 分を超えたため打ち切り)")
    else:
        line = f"sub {sub:3d}: {prev_s} -> error: {res['error']}"

    print("\r" + line.ljust(110), flush=True)


targets = sorted(dissim)          # 120 名全員
results = {}
queue = list(targets)
progress = {"next": 0, "done": 0}


def _flush_in_order():
    """番号順に、出せるところまで表示する。計算は止めない。"""
    while progress["next"] < len(targets) and targets[progress["next"]] in results:
        _report(results[targets[progress["next"]]])
        progress["next"] += 1


print(f"{len(targets)} 名を {MAX_WORKERS} 並列で計算します "
      f"(1 名あたり最大 {TIMEOUT_SEC // 60} 分)\n")
wall_start = time.perf_counter()

# ワーカーがメモリ不足などで落ちた場合に備えて、1 度だけ入れ直して再試行する
for attempt in (1, 2):
    if not queue:
        break
    if attempt == 2:
        print(f"\n--- ワーカーが停止したため {len(queue)} 名を再実行します ---\n")

    broken = []
    try:
        with ProcessPoolExecutor(max_workers=MAX_WORKERS,
                                 mp_context=mp.get_context("fork")) as pool:
            futures = {pool.submit(solve_one, sub): sub for sub in queue}
            for fut in as_completed(futures):
                sub = futures[fut]
                try:
                    res = fut.result()
                except BrokenProcessPool:
                    broken.append(sub)
                    continue
                except Exception as exc:
                    res = {"sub": sub, "status": "error", "limit": None, "rank": None,
                           "sec": float("nan"),
                           "error": f"{type(exc).__name__}: {exc}"}
                results[sub] = _enrich(res)
                progress["done"] += 1

                before = progress["next"]
                _flush_in_order()
                if progress["next"] == before:
                    # まだ番号順に出せない。計算は進んでいるので進捗だけ知らせる
                    waiting = targets[progress["next"]]
                    print(f"  ... 完了 {progress['done']}/{len(targets)}"
                          f"（sub {waiting} の完了待ち）".ljust(110),
                          end="\r", flush=True)
    except BrokenProcessPool:
        broken.extend(sub for sub in queue if sub not in results)

    queue = [sub for sub in broken if sub not in results]

# 2 度試しても終わらなかったものはエラー扱いにする
for sub in queue:
    results[sub] = _enrich({"sub": sub, "status": "error", "limit": None, "rank": None,
                            "sec": float("nan"),
                            "error": "worker process died (メモリ不足の可能性)"})
_flush_in_order()

elapsed = time.perf_counter() - wall_start

summary = (
    pd.DataFrame(list(results.values()))
      .reindex(columns=["sub", "status", "prev", "limit", "limit_float",
                        "changed", "rank", "sec", "error"])
      .sort_values("sub")
      .reset_index(drop=True)
)
# changed は判定不能を <NA> として残したいので nullable boolean にする
summary["changed"] = summary["changed"].astype("boolean")

counts = summary["status"].value_counts()
print(f"\n所要時間: {elapsed / 60:.1f} 分")
print("  ok      :", int(counts.get("ok", 0)))
print("  timeout :", int(counts.get("timeout", 0)))
print("  error   :", int(counts.get("error", 0)))
print("  prev と異なる :", int(summary["changed"].sum()),
      "/ 判定できた", int(summary["changed"].notna().sum()), "名")

out_path = paths.table("genmag_exact_results.csv")
summary.to_csv(out_path, index=False)
print("保存:", out_path)

summary

120 名を 8 並列で計算します (1 名あたり最大 5 分)

sub   0:  9.10000 -> 7.87500, rank: 22   [18.6s]  <<< CHANGED                                                 
sub   1: 18.00000 -> 17.0000, rank: 22   [16.7s]  <<< CHANGED                                                 
sub   2: 20.00000 -> 16.2000, rank: 20   [15.9s]  <<< CHANGED                                                 
sub   3: 11.00000 -> timeout (5 分を超えたため打ち切り)                                                                  
sub   4: 22.00000 -> 21.0000, rank: 22   [11.1s]  <<< CHANGED                                                 
sub   5: 17.00000 -> 17.0000, rank: 23   [27.7s]                                                              
sub   6: 12.00000 -> 10.0000, rank: 20   [15.2s]  <<< CHANGED                                                 
sub   7: 16.00000 -> 16.0000, rank: 23   [20.0s]                                                              
sub   8: 10.00000 -> timeout (5 分を超えたため打ち切り)                                  

,sub,status,prev,limit,limit_float,changed,rank,sec,error
0,0,ok,9.1,63/8,7.875000,True,22.0,18.631683,None
1,1,ok,18.0,17,17.000000,True,22.0,16.725410,None
2,2,ok,20.0,81/5,16.200000,True,20.0,15.905388,None
3,3,timeout,11.0,None,NaN,<NA>,NaN,300.000774,None
4,4,ok,22.0,21,21.000000,True,22.0,11.139000,None
...,...,...,...,...,...,...,...,...,...
115,115,ok,12.0,12,12.000000,False,22.0,18.430775,None
116,116,ok,11.0,157/15,10.466667,True,21.0,30.288332,None
117,117,ok,16.0,15,15.000000,True,22.0,26.520607,None
118,118,ok,17.0,13,13.000000,True,21.0,19.887508,None
